
# Rossmann Store Sales Forecasting — End-to-End Project Workbook

## Project objective

The goal of this project is to forecast daily sales for Rossmann stores using historical sales, store metadata, calendar effects, promotions, competition information, and short-term sales history.

This workbook is designed as a **GitHub-ready project narrative**, not only as a sequence of modeling cells. It documents:

- what was done;
- why each step was necessary;
- what assumptions were made;
- which approaches were tested;
- which results were later rejected after leakage auditing;
- how the final model was selected;
- and how the final Kaggle submission is generated.

---

## Final conclusion

After correcting the time-series leakage discovered during model development, the strongest valid model was:

> **CatBoost — validation RMSPE ≈ 0.1581**

The final leakage-safe comparison was:

| Model | RMSPE | Final interpretation |
|---|---:|---|
| **CatBoost** | **0.158083** | Best valid model |
| CatBoost + LightGBM blend | 0.158083 | No improvement over CatBoost |
| LightGBM | 0.234476 | Weaker after leakage-safe rebuild |
| XGBoost baseline | 0.325635 | Not competitive in corrected pipeline |

Earlier scores are documented later in the workbook, but they are explicitly separated from the final comparison because the original feature pipeline contained target leakage.



## 1. Project design and evaluation strategy

Rossmann sales forecasting is a **time-dependent regression problem**. Therefore, the modeling strategy must respect chronology.

### Why a random train/test split was not used

A random split can allow observations from later dates to influence training while earlier dates appear in validation. That does not reflect the real forecasting problem.

The project therefore uses a **time-based holdout**, with the most recent six weeks used as validation.

### Primary metric: RMSPE

Rossmann competition performance is naturally evaluated using Root Mean Square Percentage Error:

\[
RMSPE =
\sqrt{
\frac{1}{n}
\sum_{i=1}^{n}
\left(
\frac{y_i-\hat y_i}{y_i}
\right)^2
}
\]

RMSPE is useful here because errors are evaluated relative to store sales scale rather than only in absolute units.

RMSE is still reported as a secondary diagnostic metric.



## 2. Imports and data loading

The original project was developed locally. For GitHub reproducibility, the data directory is configurable.

Expected files:

- `train.csv`
- `test.csv`
- `store.csv`

A typical repository layout is:

```text
rossmann-sales-forecasting/
├── data/
│   ├── train.csv
│   ├── test.csv
│   └── store.csv
├── notebooks/
│   └── Rossmann_Project_Workbook_GitHub.ipynb
├── requirements.txt
└── README.md
```

The raw Kaggle dataset should generally not be committed if redistribution is restricted; users can download it separately and place it in `data/`.


In [ ]:

import os
from pathlib import Path

import numpy as np
import pandas as pd

# GitHub-friendly data path.
# Override with environment variable ROSSMANN_DATA_DIR if needed.
DATA_DIR = Path(os.getenv("ROSSMANN_DATA_DIR", "data"))

# Optional fallback to the original local development path.
legacy_path = Path(
    "/Users/nazanin/Documents/ML Projects/"
    "Rossmann Store Sales/rossmann-store-sales"
)

if not DATA_DIR.exists() and legacy_path.exists():
    DATA_DIR = legacy_path

train_path = DATA_DIR / "train.csv"
test_path = DATA_DIR / "test.csv"
store_path = DATA_DIR / "store.csv"

for p in [train_path, test_path, store_path]:
    if not p.exists():
        raise FileNotFoundError(
            f"Missing required file: {p}\n"
            "Set ROSSMANN_DATA_DIR or place the CSV files in ./data/"
        )

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
store_df = pd.read_csv(store_path)

print("train:", train_df.shape)
print("test :", test_df.shape)
print("store:", store_df.shape)



## 3. Initial data audit

Before engineering features, the project checks:

- missing values;
- date range;
- sales behavior on closed stores;
- zero-sales observations while stores are open;
- holiday encoding;
- missing store metadata;
- Promo2 missing-value patterns;
- missing competition distance.

### Why this matters

Several Rossmann variables have **business semantics behind their missing values**. Treating every missing value as generic noise would lose information.

Examples:

- Promo2 metadata are often missing because a store does not participate in Promo2.
- Competition opening month/year may be missing even when a competitor distance is known.
- Missing competition opening date should not automatically mean "competitor closed".


In [ ]:
# Step 1.1: Check missing values and general info of train_df
print("=== Train Dataset Info ===")
print(train_df.info())

print("\n=== Missing Values Count ===")
print(train_df.isnull().sum())

print("\n=== First 5 Rows ===")
print(train_df.head())

In [ ]:
# Step 2.1: Convert Date column to datetime format
train_df['Date'] = pd.to_datetime(train_df['Date'])

# Step 2.2: Check date range
print(f"Date range: From {train_df['Date'].min()} to {train_df['Date'].max()}")

# Step 2.3: Basic summary statistics for numerical columns
print("\n=== Numerical Features Summary ===")
print(train_df[['Sales', 'Customers', 'Open', 'Promo', 'SchoolHoliday']].describe().T)

# Step 2.4: Check unique values of StateHoliday
print("\n=== Unique values in StateHoliday ===")
print(train_df['StateHoliday'].value_counts())

In [ ]:
# Step 3.1: Check Sales when Store is closed (Open == 0)
closed_days_sales = train_df[train_df['Open'] == 0]['Sales'].sum()
print(f"Total sales on closed days (Open == 0): {closed_days_sales}")

# Step 3.2: Check open stores with zero sales
open_zero_sales = train_df[(train_df['Open'] == 1) & (train_df['Sales'] == 0)]
print(f"Number of rows where store was OPEN but Sales == 0: {len(open_zero_sales)}")

# Step 3.3: Standardize StateHoliday values (convert 0 to '0')
train_df['StateHoliday'] = train_df['StateHoliday'].astype(str).replace({'0.0': '0'})
print("\nCleaned StateHoliday value counts:")
print(train_df['StateHoliday'].value_counts())

In [ ]:
# Step 4.1: Check store_df general info and missing values
print("=== Store Dataset Info ===")
print(store_df.info())

print("\n=== Store Missing Values Count ===")
print(store_df.isnull().sum())

print("\n=== First 5 Rows of store_df ===")
print(store_df.head())

In [ ]:
# Step 5.1: Verify Promo2 null logic
promo2_zero_count = (store_df['Promo2'] == 0).sum()
promo2_null_count = store_df['Promo2SinceWeek'].isnull().sum()

print(f"Stores with Promo2 == 0: {promo2_zero_count}")
print(f"Null count in Promo2SinceWeek: {promo2_null_count}")

# Step 5.2: Inspect the 3 stores with missing CompetitionDistance
missing_comp_dist = store_df[store_df['CompetitionDistance'].isnull()]
print("\n=== Stores with missing CompetitionDistance ===")
print(missing_comp_dist[['Store', 'StoreType', 'Assortment', 'CompetitionDistance']])

In [ ]:
# Step 6.1: Merge train_df with store_df on 'Store'
merged_df = pd.merge(train_df, store_df, on='Store', how='left')

# Step 6.2: Inspect merged dataset shape and info
print("=== Merged Dataset Info ===")
print(f"Shape: {merged_df.shape}")
print("\n=== Missing values in merged dataset ===")
print(merged_df.isnull().sum())


## 4. Target-independent feature engineering

The first engineered features do **not use Sales**, so they can be created safely before model training.

### Calendar features

- Year
- Month
- ISO week
- Month name
- Weekend indicator
- Holiday indicator

### Promotion features

`IsPromo2Active` combines:

1. whether the store participates in Promo2;
2. whether the current date is after the Promo2 start week/year;
3. whether the current month belongs to the store's PromoInterval.

### Competition features

Competition handling was revised during the project after examining missing-value semantics.

The final logic is:

- preserve `CompetitionDistanceMissing`;
- replace missing distance with a deliberately large distance;
- preserve `CompetitionOpenDateMissing`;
- set `IsCompetitionOpen = 1` only when the opening date is known and has passed;
- set `IsCompetitionOpen = 0` only when the known opening date is still in the future;
- leave `IsCompetitionOpen` missing when the opening date itself is unknown.

This avoids incorrectly interpreting "unknown opening date" as "competitor not open".


In [ ]:

# Extract year, month, week number, and month name from Date
merged_df['Year'] = merged_df['Date'].dt.year
merged_df['Month'] = merged_df['Date'].dt.month
merged_df['Week'] = merged_df['Date'].dt.isocalendar().week.astype(int)
merged_df['MonthStr'] = merged_df['Date'].dt.strftime('%b')

# ---------------------------------------------------------
# Promo2 activity
# ---------------------------------------------------------
def check_promo2_month(row):
    if pd.isna(row['PromoInterval']):
        return False
    return row['MonthStr'] in row['PromoInterval'].split(',')

promo2_started = (
    (merged_df['Year'] > merged_df['Promo2SinceYear']) |
    (
        (merged_df['Year'] == merged_df['Promo2SinceYear']) &
        (merged_df['Week'] >= merged_df['Promo2SinceWeek'])
    )
)

promo2_month_match = merged_df[
    ['MonthStr', 'PromoInterval']
].apply(check_promo2_month, axis=1)

merged_df['IsPromo2Active'] = (
    promo2_started.fillna(False) &
    promo2_month_match &
    (merged_df['Promo2'] == 1)
).astype(int)

# ---------------------------------------------------------
# Competition missing-value semantics
# ---------------------------------------------------------
merged_df['CompetitionDistanceMissing'] = (
    merged_df['CompetitionDistance'].isna().astype(int)
)

max_known_comp_distance = merged_df['CompetitionDistance'].max()
far_competitor_value = (
    max_known_comp_distance * 1.5
    if pd.notna(max_known_comp_distance)
    else 100000.0
)

merged_df['CompetitionDistance'] = (
    merged_df['CompetitionDistance']
    .fillna(far_competitor_value)
)

competition_open_date = pd.to_datetime(
    {
        'year': merged_df['CompetitionOpenSinceYear'],
        'month': merged_df['CompetitionOpenSinceMonth'],
        'day': 1
    },
    errors='coerce'
)

merged_df['CompetitionOpenDateMissing'] = (
    competition_open_date.isna().astype(int)
)

# 1 = known open, 0 = known not yet open, NaN = opening date unknown
merged_df['IsCompetitionOpen'] = np.where(
    competition_open_date.notna(),
    (merged_df['Date'] >= competition_open_date).astype(int),
    np.nan
)

merged_df['HasKnownCompetitorDistance'] = (
    1 - merged_df['CompetitionDistanceMissing']
)

print("=== IsPromo2Active Value Counts ===")
print(merged_df['IsPromo2Active'].value_counts(dropna=False))

print("\n=== IsCompetitionOpen Value Counts ===")
print(merged_df['IsCompetitionOpen'].value_counts(dropna=False))

print("\nCompetitionDistance missing flag:")
print(merged_df['CompetitionDistanceMissing'].value_counts(dropna=False))


In [ ]:

# Competition distance intervals
bins = [-1, 500, 2500, 7000, float('inf')]
labels = ['Very_Close', 'Close', 'Medium', 'Far']

merged_df['CompDistance_CustomBin'] = pd.cut(
    merged_df['CompetitionDistance'],
    bins=bins,
    labels=labels
)

print("\n=== Custom Competition Distance Bin Counts ===")
print(
    merged_df['CompDistance_CustomBin']
    .value_counts(dropna=False)
)



## 5. Exploratory analysis

The EDA stage examined the relationship between sales and:

- competition distance;
- Promo2 activity;
- active competition;
- store type;
- assortment;
- weekday / weekend patterns;
- temporal and lag-derived features.

The purpose of these plots was not merely descriptive. They were used to decide which features deserved explicit representation in the modeling dataset.

The original exploratory visualization cells are retained below for reproducibility.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Summary statistics for CompetitionDistance
print("=== CompetitionDistance Summary Statistics ===")
print(store_df['CompetitionDistance'].describe())

# Visualization of CompetitionDistance distribution
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
sns.histplot(store_df['CompetitionDistance'].dropna(), bins=40, kde=True, color='teal')
plt.title('CompetitionDistance Distribution (Histogram)')

plt.subplot(1, 2, 2)
# Fixed color parameter to 'darkorange'
sns.boxplot(x=store_df['CompetitionDistance'].dropna(), color='darkorange')
plt.title('CompetitionDistance Boxplot')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization design style
sns.set_theme(style="whitegrid")

# Create a 2x2 grid for new feature visualizations
fig, axes = plt.subplots(2, 2, figsize=(16, 11))

# Filter for open stores
open_stores_df = merged_df[merged_df['Open'] == 1].copy()

# Map binary features to descriptive strings for clean seaborn legends
open_stores_df['Promo2Active_Label'] = open_stores_df['IsPromo2Active'].map({0: 'Promo2 Inactive', 1: 'Promo2 Active'})
open_stores_df['CompOpen_Label'] = open_stores_df['IsCompetitionOpen'].map({0: 'No Active Competitor', 1: 'Competitor Open'})

# 1. Average Sales by Competition Distance Bins
sns.barplot(data=open_stores_df, x='CompDistance_CustomBin', y='Sales', ax=axes[0, 0], palette='Blues_d')
axes[0, 0].set_title('1. Mean Sales by Competition Distance Bins', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Distance Category')
axes[0, 0].set_ylabel('Mean Sales')

# 2. Average Sales: Active Competitor vs No Competitor
sns.barplot(data=open_stores_df, x='CompOpen_Label', y='Sales', ax=axes[0, 1], palette='Set2')
axes[0, 1].set_title('2. Mean Sales: Competitor Open vs Inactive', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Competitor Status')
axes[0, 1].set_ylabel('Mean Sales')

# 3. Average Sales by IsPromo2Active Status
sns.barplot(data=open_stores_df, x='Promo2Active_Label', y='Sales', ax=axes[1, 0], palette='Pastel1')
axes[1, 0].set_title('3. Mean Sales by Promo2 Active Status', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Promo2 Status')
axes[1, 0].set_ylabel('Mean Sales')

# 4. Average Sales by StoreType & Promo2 Active Status
sns.barplot(data=open_stores_df, x='StoreType', y='Sales', hue='Promo2Active_Label', ax=axes[1, 1], palette='viridis')
axes[1, 1].set_title('4. Mean Sales by StoreType & Promo2 Activity', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Store Type')
axes[1, 1].set_ylabel('Mean Sales')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set figure size and style
plt.figure(figsize=(11, 9))
sns.set_theme(style="white")

# Include new engineered features in numerical list
numerical_cols = [
    'Sales', 'Customers', 'CompetitionDistance',
    'Promo', 'SchoolHoliday', 'DayOfWeek',
    'Promo2', 'IsPromo2Active', 'IsCompetitionOpen'
]

# Calculate correlation matrix for open stores
corr_matrix = open_stores_df[numerical_cols].corr()

# Draw correlation heatmap
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap='coolwarm',
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    cbar_kws={"shrink": .8}
)

plt.title('Updated Correlation Heatmap (Including Engineered Features)', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create a pivot table for any two categorical features against mean Sales
pivot_df = open_stores_df.pivot_table(
    index='StoreType',
    columns='Assortment',
    values='Sales',
    aggfunc='mean'
)

plt.figure(figsize=(8, 5))
sns.heatmap(pivot_df, annot=True, fmt=".0f", cmap='Blues', cbar=False)
plt.title('Mean Sales: StoreType vs Assortment Interaction', fontsize=12, fontweight='bold')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create Promo_Label directly on open_stores_df
open_stores_df['Promo_Label'] = open_stores_df['Promo'].map({0: 'No Promo', 1: 'Promo'})

# Multi-panel plot to observe DayOfWeek patterns across different StoreTypes
sns.catplot(
    data=open_stores_df,
    x='DayOfWeek',
    y='Sales',
    col='StoreType',
    hue='Promo_Label',
    kind='bar',
    height=3.5,
    aspect=1.1,
    palette='Set1'
)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create clear string labels for better chart interpretation
open_stores_df['Promo_Label'] = open_stores_df['Promo'].map({0: 'No Daily Promo', 1: 'Daily Promo'})
open_stores_df['Promo2Active_Label'] = open_stores_df['IsPromo2Active'].map({0: 'Promo2 Inactive', 1: 'Promo2 Active'})

# 1. Pivot Table to inspect Mean Sales and Transaction Count across all 4 combinations
promo_interaction_pivot = open_stores_df.pivot_table(
    index='Promo_Label',
    columns='Promo2Active_Label',
    values='Sales',
    aggfunc=['mean', 'count']
)

print("=== Mean Sales & Transaction Count Across Promo Combinations ===")
print(promo_interaction_pivot)

# 2. Visualizing the interaction effect
plt.figure(figsize=(9, 6))
sns.barplot(
    data=open_stores_df,
    x='Promo_Label',
    y='Sales',
    hue='Promo2Active_Label',
    palette='Set2'
)

plt.title('Interaction Effect: Daily Promo vs Active Promo2', fontsize=12, fontweight='bold')
plt.xlabel('Short-Term Daily Promotion (Promo)')
plt.ylabel('Mean Sales')
plt.tight_layout()
plt.show()


## 6. Time-series feature engineering and the leakage audit

This was the most important methodological correction in the project.

### Initial approach

The early version created:

- `Sales_Lag_1`
- `Sales_Lag_7`
- `Sales_Rolling_Mean_7`
- `Sales_Rolling_Std_7`

However, initial missing lag values were filled using Store × DayOfWeek sales averages calculated over the full dataset, and backward filling was also used.

That means some early historical rows could indirectly receive information derived from **future Sales**.

This is target leakage.

### Why the initial XGBoost result was suspicious

Before the leakage audit, XGBoost produced:

- RMSE ≈ 968.98
- RMSPE ≈ 0.12505

This was dramatically stronger than every previous model and triggered a pipeline audit.

After rebuilding the historical features so that every target-derived fallback used only prior observations, XGBoost fell to RMSPE ≈ **0.32564**.

The earlier 0.125 score is therefore retained only as an example of why unexpectedly strong validation performance should be audited.

### Final leakage-safe logic

All Sales-derived features now use:

- `shift(1)` before rolling operations;
- shifted expanding historical means for early missing values;
- no backward fill;
- no future sales averages.

No additional long lags were added because the project intentionally avoided losing large numbers of early training rows.


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Sort chronologically per store
merged_df = (
    merged_df
    .sort_values(['Store', 'Date'])
    .reset_index(drop=True)
)

# 2. Calendar / sequence features
merged_df['IsWeekend'] = (
    merged_df['DayOfWeek'].isin([6, 7]).astype(int)
)

merged_df['IsHoliday'] = (
    merged_df['StateHoliday'].astype(str).ne('0').astype(int)
)

merged_df['WasClosedPrevDay'] = (
    merged_df.groupby('Store')['Open']
    .shift(1)
    .eq(0)
    .fillna(False)
    .astype(int)
)

merged_df['WillBeClosedNextDay'] = (
    merged_df.groupby('Store')['Open']
    .shift(-1)
    .eq(0)
    .fillna(False)
    .astype(int)
)

merged_df['WasHolidayPrevDay'] = (
    merged_df.groupby('Store')['IsHoliday']
    .shift(1)
    .fillna(0)
    .astype(int)
)

merged_df['WillBeHolidayNextDay'] = (
    merged_df.groupby('Store')['IsHoliday']
    .shift(-1)
    .fillna(0)
    .astype(int)
)

# 3. True historical Sales features
merged_df['Sales_Lag_1'] = (
    merged_df.groupby('Store')['Sales'].shift(1)
)

merged_df['Sales_Lag_7'] = (
    merged_df.groupby('Store')['Sales'].shift(7)
)

merged_df['Sales_Rolling_Mean_7'] = (
    merged_df.groupby('Store')['Sales']
    .transform(
        lambda s:
        s.shift(1)
         .rolling(7, min_periods=1)
         .mean()
    )
)

merged_df['Sales_Rolling_Std_7'] = (
    merged_df.groupby('Store')['Sales']
    .transform(
        lambda s:
        s.shift(1)
         .rolling(7, min_periods=2)
         .std()
    )
)

# 4. Leakage-safe imputation for early rows
# Every fallback below is shifted, so it uses historical Sales only.
store_dow_hist_mean = (
    merged_df.groupby(['Store', 'DayOfWeek'])['Sales']
    .transform(
        lambda s:
        s.shift(1)
         .expanding()
         .mean()
    )
)

store_hist_mean = (
    merged_df.groupby('Store')['Sales']
    .transform(
        lambda s:
        s.shift(1)
         .expanding()
         .mean()
    )
)

global_hist_mean = (
    merged_df['Sales']
    .shift(1)
    .expanding()
    .mean()
)

for col in [
    'Sales_Lag_1',
    'Sales_Lag_7',
    'Sales_Rolling_Mean_7'
]:
    merged_df[col] = (
        merged_df[col]
        .fillna(store_dow_hist_mean)
        .fillna(store_hist_mean)
        .fillna(global_hist_mean)
        .fillna(0.0)
    )

merged_df['Sales_Rolling_Std_7'] = (
    merged_df['Sales_Rolling_Std_7']
    .fillna(0.0)
)

print("=== Missing Values in Leakage-Safe Sales Features ===")
print(
    merged_df[
        [
            'Sales_Lag_1',
            'Sales_Lag_7',
            'Sales_Rolling_Mean_7',
            'Sales_Rolling_Std_7'
        ]
    ].isnull().sum()
)

# Open-store view for EDA only
open_stores_df = merged_df[
    merged_df['Open'] == 1
].copy()

engineered_cols = [
    'Sales',
    'IsWeekend',
    'WasClosedPrevDay',
    'WillBeClosedNextDay',
    'WasHolidayPrevDay',
    'WillBeHolidayNextDay',
    'Sales_Lag_1',
    'Sales_Lag_7',
    'Sales_Rolling_Mean_7',
    'Sales_Rolling_Std_7'
]

plt.figure(figsize=(10, 8))
sns.heatmap(
    open_stores_df[engineered_cols].corr(),
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    vmin=-1,
    vmax=1,
    linewidths=0.5
)
plt.title(
    'Correlation Matrix of Leakage-Safe Temporal Features',
    fontsize=12,
    fontweight='bold'
)
plt.tight_layout()
plt.show()



## 7. Test-set feature engineering

The test feature pipeline mirrors the training features while respecting one crucial constraint:

> Test Sales are unknown.

Therefore, test Sales-history variables cannot be calculated from actual future test Sales.

For the final submission pipeline:

- calendar, promotion and competition features are calculated directly;
- previous/next-day store-status and holiday variables are built from known exogenous information;
- Sales-history benchmarks for test rows are derived from the **full historical training data only**.

This is a practical benchmark approach for the current project and matches the leakage-safe validation philosophy.


In [ ]:

import pandas as pd
import numpy as np

# =========================================================
# TEST feature engineering
# =========================================================

if 'StoreType' not in test_df.columns:
    test_merged = pd.merge(
        test_df,
        store_df,
        on='Store',
        how='left'
    )
else:
    test_merged = test_df.copy()

test_merged['Date'] = pd.to_datetime(test_merged['Date'])
merged_df['Date'] = pd.to_datetime(merged_df['Date'])

# Keep original Kaggle row identifier
test_ids = test_merged['Id'].copy()

# Missing Open values: same practical rule used previously
test_merged['Open'] = (
    test_merged['Open']
    .fillna(
        (test_merged['DayOfWeek'] != 7)
        .astype(int)
    )
)

test_merged['StateHoliday'] = (
    test_merged['StateHoliday']
    .astype(str)
    .replace({'0.0': '0'})
)

# ---------------------------------------------------------
# Calendar
# ---------------------------------------------------------
test_merged['Year'] = test_merged['Date'].dt.year
test_merged['Month'] = test_merged['Date'].dt.month
test_merged['Week'] = (
    test_merged['Date']
    .dt.isocalendar()
    .week
    .astype(int)
)
test_merged['MonthStr'] = (
    test_merged['Date']
    .dt.strftime('%b')
)

test_merged['IsWeekend'] = (
    test_merged['DayOfWeek']
    .isin([6, 7])
    .astype(int)
)

test_merged['IsHoliday'] = (
    test_merged['StateHoliday']
    .astype(str)
    .ne('0')
    .astype(int)
)

# ---------------------------------------------------------
# Promo2
# ---------------------------------------------------------
promo_interval_test = (
    test_merged['PromoInterval']
    .fillna('')
)

promo_month_match_test = [
    month in interval.split(',')
    if interval else False
    for month, interval in zip(
        test_merged['MonthStr'],
        promo_interval_test
    )
]

promo2_started_test = (
    (test_merged['Year'] > test_merged['Promo2SinceYear'])
    |
    (
        (test_merged['Year'] == test_merged['Promo2SinceYear'])
        &
        (test_merged['Week'] >= test_merged['Promo2SinceWeek'])
    )
)

test_merged['IsPromo2Active'] = (
    promo2_started_test.fillna(False)
    &
    np.asarray(promo_month_match_test)
    &
    (test_merged['Promo2'] == 1)
).astype(int)

# ---------------------------------------------------------
# Competition logic
# ---------------------------------------------------------
test_merged['CompetitionDistanceMissing'] = (
    test_merged['CompetitionDistance']
    .isna()
    .astype(int)
)

max_known_comp_distance = (
    store_df['CompetitionDistance'].max()
)

far_competitor_value = (
    max_known_comp_distance * 1.5
)

test_merged['CompetitionDistance'] = (
    test_merged['CompetitionDistance']
    .fillna(far_competitor_value)
)

competition_open_date_test = pd.to_datetime(
    {
        'year':
            test_merged['CompetitionOpenSinceYear'],
        'month':
            test_merged['CompetitionOpenSinceMonth'],
        'day':
            1
    },
    errors='coerce'
)

test_merged['CompetitionOpenDateMissing'] = (
    competition_open_date_test
    .isna()
    .astype(int)
)

test_merged['IsCompetitionOpen'] = np.where(
    competition_open_date_test.notna(),
    (
        test_merged['Date']
        >= competition_open_date_test
    ).astype(int),
    np.nan
)

test_merged['HasKnownCompetitorDistance'] = (
    1 -
    test_merged['CompetitionDistanceMissing']
)

bins = [-1, 500, 2500, 7000, float('inf')]
labels = ['Very_Close', 'Close', 'Medium', 'Far']

test_merged['CompDistance_CustomBin'] = pd.cut(
    test_merged['CompetitionDistance'],
    bins=bins,
    labels=labels
)

# ---------------------------------------------------------
# Sequence features from train + test calendar only
# No Sales information from test is used.
# ---------------------------------------------------------
train_calendar = merged_df[
    [
        'Store',
        'Date',
        'Open',
        'StateHoliday'
    ]
].copy()
train_calendar['_source'] = 'train'

test_calendar = test_merged[
    [
        'Store',
        'Date',
        'Open',
        'StateHoliday'
    ]
].copy()
test_calendar['_source'] = 'test'
test_calendar['_test_index'] = test_calendar.index

calendar_timeline = pd.concat(
    [
        train_calendar,
        test_calendar
    ],
    ignore_index=True
)

calendar_timeline = (
    calendar_timeline
    .sort_values(
        ['Store', 'Date']
    )
    .reset_index(drop=True)
)

calendar_timeline['_holiday'] = (
    calendar_timeline['StateHoliday']
    .astype(str)
    .ne('0')
    .astype(int)
)

calendar_timeline['WasClosedPrevDay'] = (
    calendar_timeline.groupby('Store')['Open']
    .shift(1)
    .eq(0)
    .fillna(False)
    .astype(int)
)

calendar_timeline['WillBeClosedNextDay'] = (
    calendar_timeline.groupby('Store')['Open']
    .shift(-1)
    .eq(0)
    .fillna(False)
    .astype(int)
)

calendar_timeline['WasHolidayPrevDay'] = (
    calendar_timeline.groupby('Store')['_holiday']
    .shift(1)
    .fillna(0)
    .astype(int)
)

calendar_timeline['WillBeHolidayNextDay'] = (
    calendar_timeline.groupby('Store')['_holiday']
    .shift(-1)
    .fillna(0)
    .astype(int)
)

test_calendar_features = (
    calendar_timeline[
        calendar_timeline['_source'] == 'test'
    ]
    .sort_values('_test_index')
)

for col in [
    'WasClosedPrevDay',
    'WillBeClosedNextDay',
    'WasHolidayPrevDay',
    'WillBeHolidayNextDay'
]:
    test_merged[col] = (
        test_calendar_features[col]
        .to_numpy()
    )

# ---------------------------------------------------------
# Sales-history benchmarks from FULL TRAIN only
# This matches the validation benchmark strategy.
# ---------------------------------------------------------
full_open_train = (
    merged_df[
        merged_df['Open'] == 1
    ]
    .copy()
)

store_day_mean = (
    full_open_train
    .groupby(
        ['Store', 'DayOfWeek']
    )['Sales']
    .mean()
    .reset_index()
)

store_mean = (
    full_open_train
    .groupby('Store')['Sales']
    .mean()
    .reset_index()
)

store_std = (
    full_open_train
    .groupby('Store')['Sales']
    .std()
    .fillna(0)
    .reset_index()
)

test_final = test_merged.copy()

test_final = test_final.merge(
    store_day_mean.rename(
        columns={
            'Sales':
            'Sales_Lag_1'
        }
    ),
    on=[
        'Store',
        'DayOfWeek'
    ],
    how='left'
)

test_final = test_final.merge(
    store_day_mean.rename(
        columns={
            'Sales':
            'Sales_Lag_7'
        }
    ),
    on=[
        'Store',
        'DayOfWeek'
    ],
    how='left'
)

test_final = test_final.merge(
    store_mean.rename(
        columns={
            'Sales':
            'Sales_Rolling_Mean_7'
        }
    ),
    on='Store',
    how='left'
)

test_final = test_final.merge(
    store_std.rename(
        columns={
            'Sales':
            'Sales_Rolling_Std_7'
        }
    ),
    on='Store',
    how='left'
)

print("=== Test Feature Engineering Status ===")
print("Test shape:", test_final.shape)

print(
    test_final[
        [
            'Sales_Lag_1',
            'Sales_Lag_7',
            'Sales_Rolling_Mean_7',
            'Sales_Rolling_Std_7'
        ]
    ].isnull().sum()
)



## 8. Modeling dataset and preprocessing

The regression task is restricted to **open stores**.

### Why closed stores are treated separately

Rossmann closed stores normally have zero sales. Instead of asking the regression model to learn both:

- whether the store is closed;
- and how much an open store sells,

the model focuses on open-store sales. During final submission, closed stores are explicitly assigned `Sales = 0`.

### Excluded variables

`Customers` is not used as a predictive feature because customer count is not available for the future Kaggle test horizon and would create an unrealistic forecasting advantage.

Categorical variables are encoded for LightGBM/XGBoost through preprocessing, while CatBoost receives categorical variables natively.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# 1. Filter for Open stores
train_data = merged_df[merged_df['Open'] == 1].copy()

# 2. Time-Based Split (Last 6 weeks as Validation set)
val_cutoff = train_data['Date'].max() - pd.Timedelta(weeks=6)

train_set = train_data[train_data['Date'] <= val_cutoff].copy()
val_set = train_data[train_data['Date'] > val_cutoff].copy()

# 3. FIX DATA LEAKAGE: Recompute Validation Lags strictly from Train statistics
# Compute historical benchmarks on train_set ONLY
train_store_day_mean = train_set.groupby(['Store', 'DayOfWeek'])['Sales'].mean().reset_index()
train_rolling_mean = train_set.groupby('Store')['Sales'].mean().reset_index().rename(columns={'Sales': 'Sales_Rolling_Mean_7_bench'})
train_rolling_std = train_set.groupby('Store')['Sales'].std().fillna(0).reset_index().rename(columns={'Sales': 'Sales_Rolling_Std_7_bench'})

# Drop leaked actual lag features from val_set
sales_lag_cols = ['Sales_Lag_1', 'Sales_Lag_7', 'Sales_Rolling_Mean_7', 'Sales_Rolling_Std_7']
val_set = val_set.drop(columns=sales_lag_cols, errors='ignore')

# Merge Train historical benchmarks into Validation set (Identical to Test Pipeline)
val_set = val_set.merge(train_store_day_mean.rename(columns={'Sales': 'Sales_Lag_1'}), on=['Store', 'DayOfWeek'], how='left')
val_set = val_set.merge(train_store_day_mean.rename(columns={'Sales': 'Sales_Lag_7'}), on=['Store', 'DayOfWeek'], how='left')
val_set = val_set.merge(train_rolling_mean, on='Store', how='left').rename(columns={'Sales_Rolling_Mean_7_bench': 'Sales_Rolling_Mean_7'})
val_set = val_set.merge(train_rolling_std, on='Store', how='left').rename(columns={'Sales_Rolling_Std_7_bench': 'Sales_Rolling_Std_7'})

# 4. Separate Predictors and Target
drop_cols = ['Sales', 'Customers', 'Date', 'StateHoliday', 'PromoInterval']
X_train = train_set.drop(columns=drop_cols, errors='ignore')
y_train = train_set['Sales']

X_val = val_set.drop(columns=drop_cols, errors='ignore')
y_val = val_set['Sales']

# RMSPE Calculation Metric
def calculate_rmspe(y_true, y_pred):
    return np.sqrt(np.mean(((y_true - y_pred) / y_true) ** 2))

# 5. Build Scikit-Learn Pipeline
num_cols = X_train.select_dtypes(include=['int64', 'float64', 'int32']).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols)
])

rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('rf', RandomForestRegressor(n_estimators=100, max_depth=20, min_samples_split=5, random_state=42, n_jobs=-1))
])

# Train and Evaluate
print("Training Baseline Model without Data Leakage...")
rf_pipeline.fit(X_train, y_train)

y_val_pred = rf_pipeline.predict(X_val)

print("\n=== Clean Validation Results (No Leakage) ===")
print(f"RMSE:  {np.sqrt(mean_squared_error(y_val, y_val_pred)):.2f}")
print(f"RMSPE: {calculate_rmspe(y_val, y_val_pred):.4f}")

Training Baseline Model without Data Leakage...


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Extract preprocessor and feature names
preprocessor_fit = rf_pipeline.named_steps['preprocessor']
feature_names = preprocessor_fit.get_feature_names_out()
importances = rf_pipeline.named_steps['rf'].feature_importances_

# Clean feature names (remove transformer prefixes like num__ or cat__)
clean_feature_names = [f.split('__')[-1] for f in feature_names]

importance_df = pd.DataFrame({
    'Feature': clean_feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print("=== Top 15 Most Important Features ===")
print(importance_df.head(15))

# Plot Top 15 Features
plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df.head(15), x='Importance', y='Feature', palette='mako')
plt.title('Top 15 Feature Importances (Random Forest)', fontsize=12, fontweight='bold')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()


# 9. Model experimentation history

The project deliberately tested multiple model families.

## Random Forest

Random Forest provided an early nonlinear tree baseline.

**Observed RMSPE ≈ 0.1935**

It improved over simple expectations but was surpassed by gradient boosting.

## LightGBM baseline

Initial LightGBM:

**RMSPE ≈ 0.1849**

This confirmed that boosting was better suited to the structured feature set.

## LightGBM with log-transformed Sales

Because sales are positive and skewed, the target was transformed using:

```python
log1p(Sales)
```

Predictions were converted back using:

```python
expm1(prediction)
```

This improved the early LightGBM result to:

**RMSPE ≈ 0.1795**

## Initial tuned LightGBM

A time-aware Optuna search using RMSPE eventually reached approximately:

**RMSPE ≈ 0.1714**

These hyperparameters were later reused directly to avoid repeating an expensive search.

## MLP neural network

A multilayer perceptron was also tested using standardized numerical variables and one-hot encoded categorical variables.

Architecture:

```text
256 → 128 → 64 → 1
```

with Adam optimization and early stopping.

Result:

**RMSPE ≈ 0.2115**

The MLP was clearly inferior to the tree boosting models and was not pursued further.

## CatBoost

CatBoost was selected because it can model categorical variables natively and learn category interactions through its ordered boosting / CTR machinery.

Early baseline:

**RMSPE ≈ 0.16898**

Tuned pre-audit result:

**RMSPE ≈ 0.15716**

After the leakage-safe rebuild, CatBoost remained extremely stable:

**Final valid RMSPE ≈ 0.15808**

This stability was one of the strongest reasons for choosing CatBoost as the final model.



## 10. LightGBM — saved hyperparameters, no repeated expensive tuning

The original hyperparameter search was computationally expensive. Re-running it every time the notebook executes would make the repository impractical.

Therefore, the best previously discovered parameter set is stored and refit directly.

This is an important reproducibility choice:

- hyperparameter search is part of the experiment history;
- final notebook execution uses the saved best configuration.

After the leakage-safe rebuild, LightGBM's validation RMSPE was approximately **0.23448**, so it was no longer competitive with CatBoost.


In [ ]:
from lightgbm import LGBMRegressor

# Define LightGBM Pipeline
lgbm_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('lgbm', LGBMRegressor(
        n_estimators=500,
        learning_rate=0.03,
        num_leaves=63,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    ))
])

# Train LightGBM
print("Training LightGBM Model...")
lgbm_pipeline.fit(X_train, y_train)

# Evaluate on Clean Validation Set
y_val_pred_lgbm = lgbm_pipeline.predict(X_val)

rmse_lgbm = np.sqrt(mean_squared_error(y_val, y_val_pred_lgbm))
rmspe_lgbm = calculate_rmspe(y_val, y_val_pred_lgbm)

print("\n=== LightGBM Validation Results ===")
print(f"Random Forest RMSPE: 0.1935")
print(f"LightGBM RMSE:        {rmse_lgbm:.2f}")
print(f"LightGBM RMSPE:       {rmspe_lgbm:.4f}")

In [ ]:
import numpy as np

# 1. Transform Target with Log1p
y_train_log = np.log1p(y_train)

# 2. Train LightGBM on Log-Transformed Target
print("Training LightGBM on Log-Transformed Sales...")
lgbm_pipeline.fit(X_train, y_train_log)

# 3. Predict and Revert with Expm1
y_val_pred_log = lgbm_pipeline.predict(X_val)
y_val_pred_exp = np.expm1(y_val_pred_log)

# 4. Evaluate Metrics
rmse_log_lgbm = np.sqrt(mean_squared_error(y_val, y_val_pred_exp))
rmspe_log_lgbm = calculate_rmspe(y_val, y_val_pred_exp)

print("\n=== LightGBM with Log Target Results ===")
print(f"Previous LightGBM RMSPE: 0.1849")
print(f"Log-Transformed RMSE:     {rmse_log_lgbm:.2f}")
print(f"Log-Transformed RMSPE:    {rmspe_log_lgbm:.4f}")

## Stage 2 — LightGBM Hyperparameter Tuning (Time-Aware + RMSPE)

The previous `RandomizedSearchCV` block was replaced because its default K-Fold CV does not respect chronology and it selected parameters using MSE on `log(Sales)`, while the project is evaluated with RMSPE on the original Sales scale.

This tuning stage keeps the existing **last-6-weeks time-based validation split**, fits preprocessing on the training period only, trains LightGBM on `log1p(Sales)`, and asks Optuna to minimize **RMSPE after converting predictions back with `expm1`**.

Key design choices:
- **Optuna** explores hyperparameters more efficiently than a tiny random grid.
- **Fixed time-based validation** prevents random mixing of past and future rows during tuning.
- **RMSPE on original Sales** is the model-selection objective.
- **Early stopping** stops adding trees when validation RMSPE no longer improves.
- `subsample_freq > 0` is tuned together with `subsample`, so row subsampling is actually activated.
- Preprocessing is fit on `X_train` only and then applied to `X_val`.

> Note: after hyperparameter selection, this six-week block has effectively become a tuning/validation set. For a final unbiased performance estimate, reserve a separate later holdout or use walk-forward validation before final submission.


In [ ]:

# =========================================================
# FAST LightGBM refit using previously discovered best params
# No Optuna rerun
# =========================================================

import numpy as np
from sklearn.base import clone
from sklearn.metrics import mean_squared_error
from lightgbm import LGBMRegressor, early_stopping, log_evaluation

y_train_log = np.log1p(y_train)

tuning_preprocessor = clone(preprocessor)

X_train_tuned = tuning_preprocessor.fit_transform(
    X_train
)

X_val_tuned = tuning_preprocessor.transform(
    X_val
)

def lgbm_rmspe_from_log(
    y_true_log,
    y_pred_log
):
    y_true_sales = np.expm1(y_true_log)
    y_pred_sales = np.clip(
        np.expm1(y_pred_log),
        0,
        None
    )

    score = calculate_rmspe(
        y_true_sales,
        y_pred_sales
    )

    return 'rmspe', score, False

BEST_LGBM_PARAMS = {
    'num_leaves': 229,
    'max_depth': 12,
    'min_child_samples': 180,
    'min_split_gain': 0.0002567791466346484,
    'learning_rate': 0.03418631933050785,
    'subsample': 0.9302324264248383,
    'subsample_freq': 5,
    'colsample_bytree': 0.6004273275034597,
    'reg_alpha': 4.834483708168735e-05,
    'reg_lambda': 1.8633221909250445e-06,
    'max_bin': 63,
    'n_estimators': 3000,
    'objective': 'regression',
    'metric': 'None',
    'random_state': 42,
    'n_jobs': -1,
    'verbosity': -1
}

best_lgbm_model = LGBMRegressor(
    **BEST_LGBM_PARAMS
)

print(
    "Training LightGBM directly "
    "with saved best parameters..."
)

best_lgbm_model.fit(
    X_train_tuned,
    y_train_log,
    eval_set=[
        (
            X_val_tuned,
            np.log1p(y_val)
        )
    ],
    eval_metric=lgbm_rmspe_from_log,
    callbacks=[
        early_stopping(
            stopping_rounds=100,
            first_metric_only=True,
            verbose=False
        ),
        log_evaluation(period=0)
    ]
)

best_iteration = (
    best_lgbm_model.best_iteration_
)

y_val_pred_log_tuned = (
    best_lgbm_model.predict(
        X_val_tuned,
        num_iteration=best_iteration
    )
)

y_val_pred_tuned = np.clip(
    np.expm1(
        y_val_pred_log_tuned
    ),
    0,
    None
)

rmse_tuned = np.sqrt(
    mean_squared_error(
        y_val,
        y_val_pred_tuned
    )
)

rmspe_tuned = calculate_rmspe(
    y_val,
    y_val_pred_tuned
)

print(
    "\n=== Fast LightGBM Results ==="
)

print(
    "Best iteration:",
    best_iteration
)

print(
    f"RMSE:  {rmse_tuned:.2f}"
)

print(
    f"RMSPE: {rmspe_tuned:.8f}"
)



## 11. CatBoost — final model family

### Why CatBoost was especially appropriate

The dataset contains several categorical variables such as:

- StoreType
- Assortment
- MonthStr
- competition distance band
- Store
- DayOfWeek
- Month

Instead of one-hot encoding all of them externally, CatBoost can work with categorical inputs directly.

### Missing categorical values

A specific implementation issue was discovered during development:

```text
Cannot setitem on a Categorical with a new category (__MISSING__)
```

The fix is to convert pandas categorical columns to string **before** filling missing values:

```python
.astype("string").fillna("__MISSING__").astype(str)
```

This correction is implemented in the final CatBoost pipeline.

### Final valid result

After the leakage-safe rebuild:

**CatBoost RMSPE ≈ 0.158083**

This became the final selected model.



## Stage 3 — CatBoost Baseline + Hyperparameter Tuning

This stage evaluates **CatBoost** as the next non-neural tabular model and tunes it from the start.

The comparison is deliberately kept fair:

- the same **last-6-weeks time-based validation split** (`X_train` / `X_val`) is reused;
- the same engineered features are reused;
- the target is again `log1p(Sales)`;
- final model selection is based on **RMSPE on the original Sales scale**;
- the current champion remains **Tuned LightGBM RMSPE ≈ 0.1714**.

### Why CatBoost is treated differently

CatBoost can consume categorical variables directly, so we **do not one-hot encode them**. In particular, `Store` is explicitly treated as categorical even though it is stored as an integer: store ID is an identifier, not a continuous quantity.

Categorical columns include the existing object/category columns plus selected discrete identifiers/calendar variables when present:

`Store`, `DayOfWeek`, `Month`

CatBoost handles these categories internally when building its ordered target statistics / categorical combinations.

### Tuning strategy

The stage has two parts:

1. **CatBoost baseline** with sensible defaults and early stopping.
2. **Optuna fine-tuning**, where each trial trains a CatBoost model and returns validation RMSPE on the original Sales scale.

CatBoost's training loss / early-stopping metric is RMSE on `log1p(Sales)`, while Optuna chooses the final hyperparameters using the project's actual RMSPE. This keeps training stable while model selection stays aligned with the Rossmann objective.

The main hyperparameters searched are:

- `depth`: tree complexity;
- `learning_rate`: contribution of each boosting step;
- `l2_leaf_reg`: L2 regularization;
- `random_strength`: randomness applied when scoring splits;
- `bagging_temperature`: strength of Bayesian row sampling;
- `border_count`: number of numeric feature bins;
- `one_hot_max_size`: threshold below which small categorical variables may be one-hot encoded internally;
- `max_ctr_complexity`: maximum complexity of categorical feature combinations.

> Important: this still uses the same six-week validation block for tuning. Before final submission, use a separate untouched holdout or walk-forward validation for an unbiased final comparison.


In [ ]:

# =========================================================
# FAST CatBoost validation using saved best parameters
# No Optuna rerun
# =========================================================

import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error
from catboost import (
    CatBoostRegressor,
    Pool
)

X_train_cat = X_train.copy()
X_val_cat = X_val.copy()

native_object_cats = (
    X_train_cat
    .select_dtypes(
        include=[
            'object',
            'category'
        ]
    )
    .columns
    .tolist()
)

forced_cats = [
    c
    for c in [
        'Store',
        'DayOfWeek',
        'Month'
    ]
    if c in X_train_cat.columns
]

catboost_cat_cols = list(
    dict.fromkeys(
        native_object_cats
        +
        forced_cats
    )
)

# IMPORTANT:
# convert category -> pandas string BEFORE filling missing values
for col in catboost_cat_cols:

    X_train_cat[col] = (
        X_train_cat[col]
        .astype('string')
        .fillna('__MISSING__')
        .astype(str)
    )

    X_val_cat[col] = (
        X_val_cat[col]
        .astype('string')
        .fillna('__MISSING__')
        .astype(str)
    )

for col in X_train_cat.columns:

    if col not in catboost_cat_cols:

        X_train_cat[col] = pd.to_numeric(
            X_train_cat[col],
            errors='coerce'
        )

        X_val_cat[col] = pd.to_numeric(
            X_val_cat[col],
            errors='coerce'
        )

train_pool = Pool(
    X_train_cat,
    label=np.log1p(
        y_train.to_numpy()
    ),
    cat_features=catboost_cat_cols
)

val_pool = Pool(
    X_val_cat,
    label=np.log1p(
        y_val.to_numpy()
    ),
    cat_features=catboost_cat_cols
)

BEST_CAT_PARAMS = {
    'depth':
        10,

    'learning_rate':
        0.11172333193973306,

    'l2_leaf_reg':
        6.469870699850824,

    'random_strength':
        0.01653693718282442,

    'bootstrap_type':
        'Bayesian',

    'bagging_temperature':
        0.48836057003191935,

    'border_count':
        32,

    'one_hot_max_size':
        2,

    'max_ctr_complexity':
        3,

    'loss_function':
        'RMSE',

    'eval_metric':
        'RMSE',

    'random_seed':
        42,

    'thread_count':
        -1,

    'verbose':
        False,

    'allow_writing_files':
        False
}

best_cat_model = CatBoostRegressor(
    iterations=3500,
    **BEST_CAT_PARAMS
)

print(
    "Training CatBoost directly "
    "with saved best parameters..."
)

best_cat_model.fit(
    train_pool,
    eval_set=val_pool,
    use_best_model=True,
    early_stopping_rounds=100,
    verbose=False
)

best_cat_iteration = (
    best_cat_model.get_best_iteration()
)

y_val_pred_cat_log = (
    best_cat_model.predict(
        val_pool
    )
)

y_val_pred_cat = np.clip(
    np.expm1(
        y_val_pred_cat_log
    ),
    0,
    None
)

rmse_cat_tuned = np.sqrt(
    mean_squared_error(
        y_val.to_numpy(),
        y_val_pred_cat
    )
)

rmspe_cat_tuned = (
    calculate_rmspe(
        y_val.to_numpy(),
        y_val_pred_cat
    )
)

print(
    "\n=== Leakage-Safe CatBoost Results ==="
)

print(
    "Best iteration:",
    best_cat_iteration
)

print(
    f"RMSE:  {rmse_cat_tuned:.2f}"
)

print(
    f"RMSPE: {rmspe_cat_tuned:.8f}"
)



### How to interpret the CatBoost result

After this cell finishes, keep these outputs:

- **CatBoost baseline RMSPE**
- **Best Optuna CatBoost RMSPE**
- **Best CatBoost Parameters**
- **Best iteration**
- **Tuned CatBoost RMSPE**
- **Difference vs tuned LightGBM**

Decision rule:

- **CatBoost < 0.1714** → CatBoost becomes the new single-model champion.
- **CatBoost very close to 0.1714** → keep both; their prediction errors may complement each other in a blend.
- **CatBoost materially worse** → keep LightGBM as champion and move effort toward feature engineering / XGBoost rather than over-tuning CatBoost on the same six-week block.

Even if CatBoost is slightly worse individually, do not discard it before testing a **LightGBM + CatBoost blend**, because model diversity can reduce RMSPE.



# 12. XGBoost sanity test and why it was rejected

XGBoost was initially tested as another gradient boosting implementation.

### Pre-audit result

- RMSE ≈ 968.98
- RMSPE ≈ 0.12505

This result was far better than every other model and therefore treated as suspicious rather than immediately accepted.

### Leakage-safe rerun

After rebuilding the historical features:

- RMSE ≈ 1813.51
- RMSPE ≈ 0.325635

Therefore, XGBoost was not tuned further.

This was a useful project lesson:

> A dramatic metric improvement is not automatically evidence of a superior model. It can also be evidence that the validation pipeline is leaking information.



# 13. Ensemble experiments

Before the leakage audit, a weighted blend of CatBoost and LightGBM improved the validation RMSPE from approximately:

- CatBoost: 0.15716
- Blend: 0.15540

The best Sales-space weighting was approximately:

- 76% CatBoost
- 24% LightGBM

A log-space blend was also tested and was slightly weaker.

However, after the leakage-safe rebuild the final blend result became:

**RMSPE ≈ 0.158083**

which was effectively identical to CatBoost alone.

Therefore, the final model remains **CatBoost**, because the ensemble no longer provides measurable validation benefit.



# 14. Final results — valid comparison

The final comparison below is the one that should be reported in the GitHub README and project summary.

| Model | Validation RMSPE | Status |
|---|---:|---|
| **CatBoost** | **0.158083** | **Selected final model** |
| CatBoost + LightGBM Blend | 0.158083 | No incremental gain |
| LightGBM | 0.234476 | Rejected |
| XGBoost baseline | 0.325635 | Rejected |

## Archived pre-audit experiments

These results describe the development path but **must not be directly compared with the final leakage-safe scores**.

| Experiment | RMSPE | Interpretation |
|---|---:|---|
| Random Forest | ~0.1935 | Early nonlinear baseline |
| LightGBM baseline | ~0.1849 | Better tree boosting baseline |
| LightGBM + log target | ~0.1795 | Improved target treatment |
| Tuned LightGBM | ~0.1714 | Best early LightGBM |
| MLP | ~0.2115 | Underperformed boosting |
| CatBoost baseline | ~0.16898 | Strong categorical model |
| Tuned CatBoost | ~0.15716 | Very strong pre-audit model |
| CatBoost/LGBM blend | ~0.15540 | Best pre-audit ensemble |
| XGBoost baseline | ~0.12505 | Rejected after leakage audit |

The main conclusion is not simply that CatBoost won. The stronger conclusion is that **CatBoost remained strong even after the validation pipeline was made stricter**, while several other apparent gains disappeared.



# 15. Final CatBoost training and Kaggle submission

For the final submission:

1. the six-week validation holdout is returned to the training pool;
2. all available **open-store** training observations are used;
3. the CatBoost iteration count selected during validation is reused;
4. the model is trained once on the complete open-store history;
5. test predictions are generated;
6. closed stores are assigned exactly zero sales;
7. predictions are saved as:

```text
Id,Sales
```

This avoids using validation predictions as submission predictions and ensures the final model benefits from the full historical training sample.



# Final CatBoost Training and Kaggle Submission

This section uses the validated CatBoost hyperparameters from the previous stage.

For the final model:

- the six-week validation holdout is returned to the training data;
- all available **open-store** training observations are used;
- the selected CatBoost iteration count from validation is reused;
- test Sales-history features are derived from historical full-train statistics only;
- closed test stores receive `Sales = 0`;
- the final file contains exactly `Id` and `Sales`.


In [ ]:

# =========================================================
# 1. Build final full training matrix
# =========================================================

full_train = (
    merged_df[
        merged_df['Open'] == 1
    ]
    .copy()
    .sort_values(
        ['Store', 'Date']
    )
    .reset_index(drop=True)
)

# Reuse EXACT model feature list from validation
feature_columns = (
    X_train
    .columns
    .tolist()
)

X_full_final = (
    full_train[
        feature_columns
    ]
    .copy()
)

y_full_final = (
    full_train['Sales']
    .astype(float)
    .copy()
)

X_test_final = (
    test_final
    .reindex(
        columns=feature_columns
    )
    .copy()
)

print(
    "Full train:",
    X_full_final.shape
)

print(
    "Test:",
    X_test_final.shape
)

assert (
    list(X_full_final.columns)
    ==
    list(X_test_final.columns)
)

print(
    "Feature alignment OK."
)


In [ ]:

# =========================================================
# 2. Final CatBoost categorical preparation
# =========================================================

final_native_cats = (
    X_full_final
    .select_dtypes(
        include=[
            'object',
            'category'
        ]
    )
    .columns
    .tolist()
)

final_forced_cats = [
    c
    for c in [
        'Store',
        'DayOfWeek',
        'Month'
    ]
    if c in X_full_final.columns
]

final_cat_cols = list(
    dict.fromkeys(
        final_native_cats
        +
        final_forced_cats
    )
)

for col in final_cat_cols:

    X_full_final[col] = (
        X_full_final[col]
        .astype('string')
        .fillna('__MISSING__')
        .astype(str)
    )

    X_test_final[col] = (
        X_test_final[col]
        .astype('string')
        .fillna('__MISSING__')
        .astype(str)
    )

for col in X_full_final.columns:

    if col not in final_cat_cols:

        X_full_final[col] = (
            pd.to_numeric(
                X_full_final[col],
                errors='coerce'
            )
        )

        X_test_final[col] = (
            pd.to_numeric(
                X_test_final[col],
                errors='coerce'
            )
        )

print(
    "Final categorical columns:"
)

print(
    final_cat_cols
)


In [ ]:

# =========================================================
# 3. Resolve final iteration count
# =========================================================

if (
    'best_cat_iteration'
    in globals()
    and best_cat_iteration is not None
    and best_cat_iteration >= 0
):
    FINAL_CAT_ITERATIONS = (
        int(best_cat_iteration)
        + 1
    )

else:
    # Historical best CatBoost run:
    # best iteration index = 2794
    FINAL_CAT_ITERATIONS = 2795

print(
    "Final CatBoost iterations:",
    FINAL_CAT_ITERATIONS
)


In [ ]:

# =========================================================
# 4. Train final CatBoost on ALL open-store training rows
# =========================================================

final_train_pool = Pool(
    X_full_final,
    label=np.log1p(
        y_full_final
        .to_numpy()
    ),
    cat_features=final_cat_cols
)

final_test_pool = Pool(
    X_test_final,
    cat_features=final_cat_cols
)

final_cat_model = CatBoostRegressor(
    iterations=FINAL_CAT_ITERATIONS,
    **BEST_CAT_PARAMS
)

print(
    "Training FINAL CatBoost..."
)

final_cat_model.fit(
    final_train_pool,
    verbose=False
)

print(
    "Final CatBoost training finished."
)


In [ ]:

# =========================================================
# 5. Predict test Sales
# =========================================================

final_pred_log = (
    final_cat_model
    .predict(
        final_test_pool
    )
)

final_pred_sales = np.clip(
    np.expm1(
        final_pred_log
    ),
    0,
    None
)

# Closed stores should predict exactly zero.
closed_mask = (
    test_final['Open']
    .to_numpy()
    == 0
)

final_pred_sales[
    closed_mask
] = 0.0

print(
    "Prediction count:",
    len(final_pred_sales)
)

print(
    "Min:",
    float(
        np.min(
            final_pred_sales
        )
    )
)

print(
    "Max:",
    float(
        np.max(
            final_pred_sales
        )
    )
)

print(
    "Mean:",
    float(
        np.mean(
            final_pred_sales
        )
    )
)


In [ ]:

# =========================================================
# 6. Create Kaggle submission.csv
# =========================================================

submission = pd.DataFrame(
    {
        'Id':
            test_final[
                'Id'
            ].to_numpy(),

        'Sales':
            final_pred_sales
    }
)

submission = (
    submission
    .sort_values('Id')
    .reset_index(drop=True)
)

assert (
    submission.shape[0]
    ==
    test_df.shape[0]
)

assert (
    submission['Id']
    .is_unique
)

assert (
    submission['Sales']
    .isna()
    .sum()
    == 0
)

assert (
    (
        submission[
            'Sales'
        ]
        < 0
    ).sum()
    == 0
)

output_file = (
    'submission_catboost_final.csv'
)

submission.to_csv(
    output_file,
    index=False
)

print(
    submission.head(10)
)

print(
    "\nShape:",
    submission.shape
)

print(
    "\nClosed stores with non-zero Sales:",
    int(
        (
            final_pred_sales[
                closed_mask
            ]
            != 0
        ).sum()
    )
)

print(
    "\nSaved:",
    output_file
)



## Final checks before Kaggle upload

The final output is ready only if:

- submission shape equals the original test row count;
- `Id` is unique;
- `Sales` contains no missing or negative values;
- closed stores have exactly zero Sales;
- the output filename is `submission_catboost_final.csv`.

Do not use validation predictions (`y_val_pred_cat`) as the Kaggle submission. The final model above is retrained on the full open-store training history.



# 16. Key lessons from the project

## 1. Validation design can matter more than model complexity

The largest apparent performance jump in the project came from XGBoost, but the improvement disappeared after leakage was removed.

## 2. Time-series target features require strict chronological construction

Any feature based on Sales must be computed from information available **before the prediction date**.

Safe patterns include:

```python
shift(1)
rolling(...)
expanding(...)
```

Unsafe patterns include full-dataset target means and backward filling of target-derived features.

## 3. Missing values have business meaning

Competition and promotion missingness should be interpreted rather than mechanically imputed.

## 4. CatBoost was robust

CatBoost remained near 0.158 RMSPE both before and after the stricter leakage audit, while other models deteriorated substantially.

## 5. Ensembles only help when component errors are complementary

The earlier CatBoost/LightGBM blend helped slightly, but after the corrected pipeline LightGBM became too weak to add value.

## 6. Expensive tuning should be separated from reproducible training

The final repository stores the best discovered hyperparameters rather than re-running long Optuna searches on every execution.



# 17. Recommended GitHub repository documentation

A concise README can report the project as:

> Developed an end-to-end Rossmann daily sales forecasting pipeline using time-aware validation, leakage-safe lag engineering, business-aware missing-value handling, LightGBM, CatBoost, XGBoost and ensemble experiments. A validation audit identified target leakage in early lag imputation, after which the pipeline was rebuilt using shifted historical statistics only. CatBoost remained the strongest valid model with RMSPE ≈ 0.1581 and was selected for the final submission.

## Suggested repository tags

`machine-learning` `time-series` `forecasting` `catboost` `lightgbm` `xgboost` `feature-engineering` `kaggle` `retail-analytics`
